# Customer Deduplication

Deduplicate customer records using email addresses and create a golden source ID.
This assigns a single `GOLDEN_ID` to each unique email, keeping the most recent record as the master.

**Lineage:** `CUSTOMERS` → `CUSTOMERS_GOLDEN`

In [ ]:
%%sql -r create_lineage_table
-- Create lineage tracking table if it doesn't exist
CREATE TABLE IF NOT EXISTS LANSDOWNEPARTNERS_DB.CORE.PIPELINE_LINEAGE (
    LINEAGE_ID NUMBER AUTOINCREMENT,
    RUN_ID VARCHAR(100),
    NOTEBOOK_NAME VARCHAR(200),
    STEP_NAME VARCHAR(200),
    SOURCE_OBJECT VARCHAR(500),
    TARGET_OBJECT VARCHAR(500),
    OPERATION VARCHAR(50),
    ROW_COUNT NUMBER,
    EXECUTION_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    STATUS VARCHAR(20) DEFAULT 'SUCCESS',
    ERROR_MESSAGE VARCHAR(2000),
    DURATION_SECONDS NUMBER
)

In [ ]:
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())[:8]
notebook_name = '01_deduplicate_customers'
start_time = datetime.now()
print(f"Pipeline Run ID: {run_id}")

## Duplicate Analysis

In [ ]:
%%sql -r duplicate_analysis
SELECT EMAIL, COUNT(*) AS duplicate_count
FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS
WHERE EMAIL IS NOT NULL
GROUP BY EMAIL
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC

## Create Golden Source Table

For each unique email, assign a `GOLDEN_ID` using `DENSE_RANK()`. The master record is the one with the most recent `RELATIONSHIP_START_DATE`.

In [ ]:
%%sql -r create_golden_source
CREATE OR REPLACE TABLE LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN AS
WITH ranked AS (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY LOWER(EMAIL)) AS GOLDEN_ID,
        ROW_NUMBER() OVER (
            PARTITION BY LOWER(EMAIL)
            ORDER BY RELATIONSHIP_START_DATE DESC NULLS LAST, CUSTOMER_ID DESC
        ) AS row_rank
    FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS
    WHERE EMAIL IS NOT NULL
)
SELECT
    GOLDEN_ID,
    CUSTOMER_ID,
    FULL_NAME,
    COMPANY_NAME,
    INVESTOR_TYPE,
    REGION,
    COUNTRY,
    AUM_COMMITMENT_GBP,
    RELATIONSHIP_START_DATE,
    RELATIONSHIP_MANAGER,
    RISK_PROFILE,
    EMAIL,
    STATUS,
    CASE WHEN row_rank = 1 THEN TRUE ELSE FALSE END AS IS_MASTER_RECORD
FROM ranked
ORDER BY GOLDEN_ID, row_rank

In [ ]:
%%sql -r verify_golden
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT GOLDEN_ID) AS unique_customers,
    COUNT(*) - COUNT(DISTINCT GOLDEN_ID) AS duplicates_identified,
    SUM(CASE WHEN IS_MASTER_RECORD THEN 1 ELSE 0 END) AS master_records
FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN

## Record Lineage

In [ ]:
from snowflake.snowpark.context import get_active_session
from datetime import datetime

session = get_active_session()
end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

# Get row count from golden table
row_count = session.sql("SELECT COUNT(*) AS cnt FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN").collect()[0]['CNT']

# Record lineage entry
session.sql(f"""
    INSERT INTO LANSDOWNEPARTNERS_DB.CORE.PIPELINE_LINEAGE 
    (RUN_ID, NOTEBOOK_NAME, STEP_NAME, SOURCE_OBJECT, TARGET_OBJECT, OPERATION, ROW_COUNT, STATUS, DURATION_SECONDS)
    VALUES (
        '{run_id}',
        '{notebook_name}',
        'deduplicate_by_email',
        'LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS',
        'LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN',
        'CREATE TABLE AS SELECT',
        {row_count},
        'SUCCESS',
        {duration}
    )
""").collect()

print(f"Lineage recorded: CUSTOMERS -> CUSTOMERS_GOLDEN ({row_count} rows, {duration}s)")